# Gold analytics

Reads the Silver Delta tables and builds the Gold date/customer/product dimensions and sales facts. Run after the Silver transformations have completed.

In [ ]:
from pyspark.sql import functions as F

customers = spark.table("adb_adventureworks_dev.silver.customer")
headers = spark.table("adb_adventureworks_dev.silver.salesorderheader")
details = spark.table("adb_adventureworks_dev.silver.salesorderdetail")
products = spark.table("adb_adventureworks_dev.silver.product")

print("Silver inputs loaded:", customers.count(), headers.count(), details.count(), products.count())


In [ ]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adb_adventureworks_dev.gold
""")

print("Gold schema is ready.")

In [ ]:
from pyspark.sql import functions as F

date_range = headers.select(
    F.to_date(F.min("OrderDate")).alias("StartDate"),
    F.to_date(F.max("OrderDate")).alias("EndDate")
).first()

dim_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(date_range["StartDate"]),
                F.lit(date_range["EndDate"]),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("FullDate")
    )
    .select(
        F.date_format("FullDate", "yyyyMMdd")
         .cast("int").alias("DateKey"),
        F.col("FullDate"),
        F.dayofmonth("FullDate").alias("Day"),
        F.month("FullDate").alias("Month"),
        F.date_format("FullDate", "MMMM").alias("MonthName"),
        F.quarter("FullDate").alias("Quarter"),
        F.year("FullDate").alias("Year"),
        F.dayofweek("FullDate").alias("DayOfWeek"),
        F.date_format("FullDate", "EEEE").alias("DayName"),
        (F.dayofweek("FullDate").isin(1, 7))
         .alias("WeekendFlag")
    )
)

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("adb_adventureworks_dev.gold.dim_date")
)

print(
    "DimDate rows:",
    spark.table("adb_adventureworks_dev.gold.dim_date").count()
)

display(
    spark.table("adb_adventureworks_dev.gold.dim_date")
    .orderBy("FullDate")
    .limit(10)
)

In [ ]:
dim_date_check = spark.table(
    "adb_adventureworks_dev.gold.dim_date"
)

print("Total rows:", dim_date_check.count())

print(
    "Distinct DateKeys:",
    dim_date_check.select("DateKey").distinct().count()
)

print(
    "Duplicate DateKeys:",
    dim_date_check.count()
    - dim_date_check.select("DateKey").distinct().count()
)

print(
    "Null DateKeys:",
    dim_date_check.filter(F.col("DateKey").isNull()).count()
)

In [ ]:
customer_check = customers.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("CustomerID").alias("DistinctCustomerIDs"),
    F.sum(
        F.when(F.col("CustomerID").isNull(), 1).otherwise(0)
    ).alias("NullCustomerIDs")
)

customer_check.show()

In [ ]:
customers.agg(
    F.sum(
        F.when(F.col("AccountNumber").isNull(), 1).otherwise(0)
    ).alias("NullAccountNumbers"),
    F.countDistinct("AccountNumber").alias("DistinctAccountNumbers"),
    F.count("*").alias("TotalRows")
).show()

In [ ]:
customers.agg(
    F.sum(
        F.when(
            F.col("PersonID").isNotNull()
            & F.col("StoreID").isNull(), 1
        ).otherwise(0)
    ).alias("PersonOnly"),

    F.sum(
        F.when(
            F.col("PersonID").isNull()
            & F.col("StoreID").isNotNull(), 1
        ).otherwise(0)
    ).alias("StoreOnly"),

    F.sum(
        F.when(
            F.col("PersonID").isNotNull()
            & F.col("StoreID").isNotNull(), 1
        ).otherwise(0)
    ).alias("PersonAndStore"),

    F.sum(
        F.when(
            F.col("PersonID").isNull()
            & F.col("StoreID").isNull(), 1
        ).otherwise(0)
    ).alias("Neither")
).show()

In [ ]:
dim_customer = customers.select(
    "CustomerID",
    "AccountNumber",
    "PersonID",
    "StoreID",
    "TerritoryID",
    "ModifiedDate"
)

(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("adb_adventureworks_dev.gold.dim_customer")
)

print(
    "DimCustomer rows:",
    spark.table("adb_adventureworks_dev.gold.dim_customer").count()
)

display(
    spark.table("adb_adventureworks_dev.gold.dim_customer")
    .orderBy("CustomerID")
    .limit(10)
)

In [ ]:
dim_customer_check = spark.table(
    "adb_adventureworks_dev.gold.dim_customer"
)

total_rows = dim_customer_check.count()

distinct_keys = (
    dim_customer_check
    .select("CustomerID")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Distinct CustomerIDs:", distinct_keys)
print("Duplicate CustomerIDs:", total_rows - distinct_keys)
print(
    "Null CustomerIDs:",
    dim_customer_check.filter(
        F.col("CustomerID").isNull()
    ).count()
)

In [ ]:
products.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("ProductID").alias("DistinctProductIDs"),
    F.countDistinct("ProductNumber").alias("DistinctProductNumbers"),
    F.sum(
        F.when(F.col("ProductID").isNull(), 1).otherwise(0)
    ).alias("NullProductIDs"),
    F.sum(
        F.when(F.col("ProductNumber").isNull(), 1).otherwise(0)
    ).alias("NullProductNumbers"),
    F.sum(
        F.when(F.col("Name").isNull(), 1).otherwise(0)
    ).alias("NullProductNames")
).show()

In [ ]:
products.agg(
    F.sum(
        F.when(F.col("ProductSubcategoryID").isNull(), 1).otherwise(0)
    ).alias("NullSubcategoryIDs"),
    F.sum(
        F.when(F.col("ProductModelID").isNull(), 1).otherwise(0)
    ).alias("NullModelIDs"),
    F.sum(
        F.when(F.col("ListPrice").isNull(), 1).otherwise(0)
    ).alias("NullListPrices"),
    F.sum(
        F.when(F.col("StandardCost").isNull(), 1).otherwise(0)
    ).alias("NullStandardCosts"),
    F.sum(
        F.when(F.col("Color").isNull(), 1).otherwise(0)
    ).alias("NullColors"),
    F.sum(
        F.when(F.col("ProductLine").isNull(), 1).otherwise(0)
    ).alias("NullProductLines")
).show()

In [ ]:
products.select(
    "ProductLine",
    "Class",
    "Style",
    "SizeUnitMeasureCode",
    "WeightUnitMeasureCode"
).distinct().orderBy(
    "ProductLine",
    "Class",
    "Style"
).show(100, truncate=False)

products.agg(
    F.min("SellStartDate").alias("EarliestSellStart"),
    F.max("SellStartDate").alias("LatestSellStart"),
    F.sum(
        F.when(F.col("SellEndDate").isNotNull(), 1).otherwise(0)
    ).alias("ProductsWithSellEndDate"),
    F.sum(
        F.when(F.col("DiscontinuedDate").isNotNull(), 1).otherwise(0)
    ).alias("DiscontinuedProducts")
).show()

In [ ]:
dim_product = products.select(
    "ProductID",
    "ProductNumber",
    "Name",
    "MakeFlag",
    "FinishedGoodsFlag",
    "Color",
    "SafetyStockLevel",
    "ReorderPoint",
    "StandardCost",
    "ListPrice",
    "Size",
    "SizeUnitMeasureCode",
    "WeightUnitMeasureCode",
    "Weight",
    "DaysToManufacture",
    "ProductLine",
    "Class",
    "Style",
    "ProductSubcategoryID",
    "ProductModelID",
    "SellStartDate",
    "SellEndDate",
    "DiscontinuedDate",
    "ModifiedDate"
)

(
    dim_product.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("adb_adventureworks_dev.gold.dim_product")
)

print(
    "DimProduct rows:",
    spark.table("adb_adventureworks_dev.gold.dim_product").count()
)

display(
    spark.table("adb_adventureworks_dev.gold.dim_product")
    .orderBy("ProductID")
    .limit(10)
)

In [ ]:
dim_product_check = spark.table(
    "adb_adventureworks_dev.gold.dim_product"
)

total_rows = dim_product_check.count()

distinct_keys = (
    dim_product_check
    .select("ProductID")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Distinct ProductIDs:", distinct_keys)
print("Duplicate ProductIDs:", total_rows - distinct_keys)

print(
    "Null ProductIDs:",
    dim_product_check.filter(
        F.col("ProductID").isNull()
    ).count()
)

In [ ]:
details.agg(
    F.count("*").alias("TotalLines"),
    F.sum("OrderQty").alias("TotalQuantity"),
    F.sum("LineTotal").alias("TotalLineSales"),
    F.sum("UnitPriceDiscount").alias("SumOfUnitPriceDiscount"),
    F.min("LineTotal").alias("MinimumLineTotal"),
    F.max("LineTotal").alias("MaximumLineTotal")
).show(truncate=False)

In [ ]:
gold_dates = spark.table(
    "adb_adventureworks_dev.gold.dim_date"
)

unmatched_order_dates = (
    headers
    .select(F.to_date("OrderDate").alias("OrderDate"))
    .distinct()
    .join(
        gold_dates.select(
            F.col("FullDate").alias("OrderDate")
        ),
        on="OrderDate",
        how="left_anti"
    )
)

print(
    "Distinct order dates without a DimDate match:",
    unmatched_order_dates.count()
)

In [ ]:
dim_date = spark.table(
    "adb_adventureworks_dev.gold.dim_date"
)

dim_customer = spark.table(
    "adb_adventureworks_dev.gold.dim_customer"
)

dim_product = spark.table(
    "adb_adventureworks_dev.gold.dim_product"
)

fact_sales_line = (
    details.alias("d")
    .join(
        headers.alias("h"),
        F.col("d.SalesOrderID") == F.col("h.SalesOrderID"),
        "inner"
    )
    .join(
        dim_customer.alias("c"),
        F.col("h.CustomerID") == F.col("c.CustomerID"),
        "inner"
    )
    .join(
        dim_product.alias("p"),
        F.col("d.ProductID") == F.col("p.ProductID"),
        "inner"
    )
    .join(
        dim_date.alias("dt"),
        F.to_date(F.col("h.OrderDate")) == F.col("dt.FullDate"),
        "inner"
    )
    .select(
        F.col("d.SalesOrderID"),
        F.col("d.SalesOrderDetailID"),
        F.col("dt.DateKey").alias("OrderDateKey"),
        F.col("c.CustomerID").alias("CustomerKey"),
        F.col("p.ProductID").alias("ProductKey"),
        F.col("d.OrderQty"),
        F.col("d.UnitPrice"),
        F.col("d.UnitPriceDiscount"),
        F.col("d.LineTotal"),
        F.col("d.ModifiedDate")
    )
)

print("FactSalesLine rows:", fact_sales_line.count())

display(
    fact_sales_line
    .orderBy("SalesOrderID", "SalesOrderDetailID")
    .limit(10)
)

In [ ]:
fact_sales_line.agg(
    F.count("*").alias("TotalLines"),
    F.sum("OrderQty").alias("TotalQuantity"),
    F.sum("LineTotal").alias("TotalLineSales"),
    F.sum("UnitPriceDiscount").alias("SumOfUnitPriceDiscount"),
    F.min("LineTotal").alias("MinimumLineTotal"),
    F.max("LineTotal").alias("MaximumLineTotal")
).show(truncate=False)

In [ ]:
(
    fact_sales_line.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("adb_adventureworks_dev.gold.fact_sales_line")
)

print("FactSalesLine saved successfully.")

In [ ]:
saved_fact = spark.table(
    "adb_adventureworks_dev.gold.fact_sales_line"
)

saved_fact.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct(
        F.struct("SalesOrderID", "SalesOrderDetailID")
    ).alias("DistinctOrderLines"),
    F.sum("OrderQty").alias("TotalQuantity"),
    F.sum("LineTotal").alias("TotalLineSales")
).show(truncate=False)

In [ ]:
from pyspark.sql import functions as F

saved_fact.select(
    F.sum(F.col("OrderDateKey").isNull().cast("int")).alias("NullOrderDateKey"),
    F.sum(F.col("CustomerKey").isNull().cast("int")).alias("NullCustomerKey"),
    F.sum(F.col("ProductKey").isNull().cast("int")).alias("NullProductKey")
).show()

print("Rows with unmatched date keys:",
      saved_fact.join(
          dim_date,
          saved_fact.OrderDateKey == dim_date.DateKey,
          "left_anti"
      ).count())

print("Rows with unmatched customer keys:",
      saved_fact.join(
          dim_customer,
          saved_fact.CustomerKey == dim_customer.CustomerID,
          "left_anti"
      ).count())

print("Rows with unmatched product keys:",
      saved_fact.join(
          dim_product,
          saved_fact.ProductKey == dim_product.ProductID,
          "left_anti"
      ).count())

In [ ]:
headers.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrders"),
    F.sum(F.col("SalesOrderID").isNull().cast("int")).alias("NullSalesOrderIDs"),
    F.sum(F.col("CustomerID").isNull().cast("int")).alias("NullCustomerIDs"),
    F.sum(F.col("OrderDate").isNull().cast("int")).alias("NullOrderDates"),
    F.sum("SubTotal").alias("TotalSubTotal"),
    F.sum("TaxAmt").alias("TotalTax"),
    F.sum("Freight").alias("TotalFreight"),
    F.sum("TotalDue").alias("TotalDue")
).show(truncate=False)

In [ ]:
headers.agg(
    F.min("OrderDate").alias("MinOrderDate"),
    F.max("OrderDate").alias("MaxOrderDate"),
    F.min("DueDate").alias("MinDueDate"),
    F.max("DueDate").alias("MaxDueDate"),
    F.min("ShipDate").alias("MinShipDate"),
    F.max("ShipDate").alias("MaxShipDate"),
    F.sum(F.col("DueDate").isNull().cast("int")).alias("NullDueDates"),
    F.sum(F.col("ShipDate").isNull().cast("int")).alias("NullShipDates")
).show(truncate=False)

In [ ]:
dim_date.select(
    F.min("FullDate").alias("MinDate"),
    F.max("FullDate").alias("MaxDate"),
    F.count("*").alias("TotalDates")
).show()

dim_date.printSchema()

In [ ]:
from datetime import date, timedelta
from pyspark.sql import functions as F

last_date = dim_date.agg(
    F.max("FullDate").alias("LastDate")
).first()["LastDate"]

extension_end_date = date(2014, 7, 12)
if last_date is not None and last_date < extension_end_date:
    new_dates = (
        spark.range(1)
        .select(
            F.explode(
                F.sequence(
                    F.lit(last_date + timedelta(days=1)),
                    F.lit(extension_end_date),
                    F.expr("INTERVAL 1 DAY")
                )
            ).alias("FullDate")
        )
        .select(
            F.date_format("FullDate", "yyyyMMdd").cast("int").alias("DateKey"),
            "FullDate",
            F.dayofmonth("FullDate").alias("Day"),
            F.month("FullDate").alias("Month"),
            F.date_format("FullDate", "MMMM").alias("MonthName"),
            F.quarter("FullDate").alias("Quarter"),
            F.year("FullDate").alias("Year"),
            F.dayofweek("FullDate").alias("DayOfWeek"),
            F.date_format("FullDate", "EEEE").alias("DayName"),
            F.dayofweek("FullDate").isin([1, 7]).alias("WeekendFlag")
        )
    )

    new_dates.write.format("delta").mode("append").saveAsTable(
        "adb_adventureworks_dev.gold.dim_date"
    )
    print("DimDate extended through", extension_end_date)
else:
    print("DimDate already covers the extension target; no rows appended.")


In [ ]:
dim_date = spark.table("adb_adventureworks_dev.gold.dim_date")

dim_date.agg(
    F.min("FullDate").alias("MinDate"),
    F.max("FullDate").alias("MaxDate"),
    F.count("*").alias("TotalDates"),
    F.countDistinct("DateKey").alias("DistinctDateKeys")
).show()

print(
    "Duplicate DateKeys:",
    dim_date.groupBy("DateKey")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Unmatched OrderDates:",
    headers.join(
        dim_date,
        F.to_date(headers.OrderDate) == dim_date.FullDate,
        "left_anti"
    ).count()
)

print(
    "Unmatched DueDates:",
    headers.join(
        dim_date,
        F.to_date(headers.DueDate) == dim_date.FullDate,
        "left_anti"
    ).count()
)

print(
    "Unmatched ShipDates:",
    headers.join(
        dim_date,
        F.to_date(headers.ShipDate) == dim_date.FullDate,
        "left_anti"
    ).count()
)

In [ ]:
fact_sales_order = (
    headers.alias("h")
    .join(
        dim_customer.alias("c"),
        F.col("h.CustomerID") == F.col("c.CustomerID"),
        "inner"
    )
    .join(
        dim_date.alias("od"),
        F.to_date(F.col("h.OrderDate")) == F.col("od.FullDate"),
        "inner"
    )
    .join(
        dim_date.alias("dd"),
        F.to_date(F.col("h.DueDate")) == F.col("dd.FullDate"),
        "inner"
    )
    .join(
        dim_date.alias("sd"),
        F.to_date(F.col("h.ShipDate")) == F.col("sd.FullDate"),
        "inner"
    )
    .select(
        F.col("h.SalesOrderID"),
        F.col("h.SalesOrderNumber"),
        F.col("od.DateKey").alias("OrderDateKey"),
        F.col("dd.DateKey").alias("DueDateKey"),
        F.col("sd.DateKey").alias("ShipDateKey"),
        F.col("c.CustomerID").alias("CustomerKey"),
        F.col("h.Status"),
        F.col("h.OnlineOrderFlag"),
        F.col("h.SubTotal"),
        F.col("h.TaxAmt"),
        F.col("h.Freight"),
        F.col("h.TotalDue"),
        F.col("h.ModifiedDate")
    )
)

print("FactSalesOrder rows:", fact_sales_order.count())

fact_sales_order.show(10, truncate=False)

In [ ]:
fact_sales_order.agg(
    F.count("*").alias("TotalOrders"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrders"),
    F.sum("SubTotal").alias("TotalSubTotal"),
    F.sum("TaxAmt").alias("TotalTax"),
    F.sum("Freight").alias("TotalFreight"),
    F.sum("TotalDue").alias("TotalDue")
).show(truncate=False)

In [ ]:
(
    fact_sales_order.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("adb_adventureworks_dev.gold.fact_sales_order")
)

print("FactSalesOrder saved successfully.")

In [ ]:
saved_order_fact = spark.table(
    "adb_adventureworks_dev.gold.fact_sales_order"
)

saved_order_fact.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrders"),
    F.sum(F.col("SalesOrderID").isNull().cast("int")).alias("NullOrderIDs"),
    F.sum(F.col("CustomerKey").isNull().cast("int")).alias("NullCustomerKeys"),
    F.sum(F.col("OrderDateKey").isNull().cast("int")).alias("NullOrderDateKeys"),
    F.sum(F.col("DueDateKey").isNull().cast("int")).alias("NullDueDateKeys"),
    F.sum(F.col("ShipDateKey").isNull().cast("int")).alias("NullShipDateKeys")
).show(truncate=False)

print(
    "Duplicate SalesOrderIDs:",
    saved_order_fact.groupBy("SalesOrderID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Unmatched CustomerKeys:",
    saved_order_fact.join(
        dim_customer,
        saved_order_fact.CustomerKey == dim_customer.CustomerID,
        "left_anti"
    ).count()
)

for key in ["OrderDateKey", "DueDateKey", "ShipDateKey"]:
    unmatched = saved_order_fact.join(
        dim_date,
        saved_order_fact[key] == dim_date.DateKey,
        "left_anti"
    ).count()

    print(f"Unmatched {key}: {unmatched}")

In [ ]:
spark.sql("""
SHOW TABLES IN adb_adventureworks_dev.gold
""").show(truncate=False)

In [ ]:
monthly_sales = (
    saved_order_fact.alias("f")
    .join(
        dim_date.alias("d"),
        F.col("f.OrderDateKey") == F.col("d.DateKey"),
        "inner"
    )
    .groupBy(
        F.col("d.Year"),
        F.col("d.Month"),
        F.col("d.MonthName")
    )
    .agg(
        F.countDistinct("f.SalesOrderID").alias("TotalOrders"),
        F.sum("f.SubTotal").alias("SalesRevenue"),
        F.sum("f.TaxAmt").alias("TotalTax"),
        F.sum("f.Freight").alias("TotalFreight"),
        F.sum("f.TotalDue").alias("TotalOrderValue")
    )
    .orderBy("Year", "Month")
)

monthly_sales.show(50, truncate=False)

In [ ]:
monthly_sales.agg(
    F.sum("TotalOrders").alias("TotalOrders"),
    F.sum("SalesRevenue").alias("TotalSalesRevenue"),
    F.sum("TotalTax").alias("TotalTax"),
    F.sum("TotalFreight").alias("TotalFreight"),
    F.sum("TotalOrderValue").alias("TotalOrderValue")
).show(truncate=False)

In [ ]:
top_products = (
    saved_fact.alias("f")
    .join(
        dim_product.alias("p"),
        F.col("f.ProductKey") == F.col("p.ProductID"),
        "inner"
    )
    .groupBy(
        F.col("p.ProductID"),
        F.col("p.ProductNumber"),
        F.col("p.Name")
    )
    .agg(
        F.sum("f.OrderQty").alias("UnitsSold"),
        F.sum("f.LineTotal").alias("SalesRevenue"),
        F.countDistinct("f.SalesOrderID").alias("Orders")
    )
    .orderBy(F.desc("SalesRevenue"))
)

top_products.show(10, truncate=False)

In [ ]:
top_customers = (
    saved_order_fact.alias("f")
    .join(
        dim_customer.alias("c"),
        F.col("f.CustomerKey") == F.col("c.CustomerID"),
        "inner"
    )
    .groupBy(
        F.col("c.CustomerID"),
        F.col("c.AccountNumber")
    )
    .agg(
        F.countDistinct("f.SalesOrderID").alias("TotalOrders"),
        F.sum("f.SubTotal").alias("SalesRevenue"),
        F.sum("f.TotalDue").alias("TotalOrderValue")
    )
    .orderBy(F.desc("SalesRevenue"))
)

top_customers.show(10, truncate=False)

In [ ]:
top_customers.agg(
    F.sum("TotalOrders").alias("TotalOrders"),
    F.sum("SalesRevenue").alias("TotalSalesRevenue"),
    F.sum("TotalOrderValue").alias("TotalOrderValue")
).show(truncate=False)

In [ ]:
top_products.agg(
    F.sum("UnitsSold").alias("TotalUnitsSold"),
    F.sum("SalesRevenue").alias("TotalLineRevenue")
).show(truncate=False)

In [ ]:
for table_name in [
    "adb_adventureworks_dev.gold.fact_sales_line",
    "adb_adventureworks_dev.gold.fact_sales_order"
]:
    print(f"\nTable: {table_name}")

    spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    ).select(
        "format",
        "numFiles",
        "sizeInBytes",
        "partitionColumns",
        "properties"
    ).show(truncate=False)

In [ ]:
monthly_sales.explain("formatted")

In [ ]:
monthly_sales_optimized = (
    saved_order_fact.alias("f")
    .join(
        dim_date.alias("d"),
        F.col("f.OrderDateKey") == F.col("d.DateKey"),
        "inner"
    )
    .groupBy(
        F.col("d.Year"),
        F.col("d.Month"),
        F.col("d.MonthName")
    )
    .agg(
        F.count("*").alias("TotalOrders"),
        F.sum("f.SubTotal").alias("SalesRevenue"),
        F.sum("f.TaxAmt").alias("TotalTax"),
        F.sum("f.Freight").alias("TotalFreight"),
        F.sum("f.TotalDue").alias("TotalOrderValue")
    )
    .orderBy("Year", "Month")
)

monthly_sales_optimized.show(50, truncate=False)

In [ ]:
monthly_sales_optimized.explain("formatted")

In [ ]:
monthly_sales_optimized.count()

In [ ]:
monthly_sales_optimized.count()

In [ ]:
aqe_settings = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.autoBroadcastJoinThreshold"
]

for setting in aqe_settings:
    try:
        value = spark.conf.get(setting)
        print(f"{setting} = {value}")
    except Exception as e:
        print(f"{setting} = Not available: {str(e).splitlines()[0]}")

In [ ]:
monthly_sales_optimized.collect()
monthly_sales_optimized.explain("formatted")

In [ ]:
monthly_sales_optimized.collect()

In [ ]:
headers = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

details = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

sales_join = (
    headers.alias("h")
    .join(
        details.alias("d"),
        F.col("h.SalesOrderID") == F.col("d.SalesOrderID"),
        "inner"
    )
    .select(
        F.col("h.SalesOrderID"),
        F.col("h.SalesOrderNumber"),
        F.col("d.SalesOrderDetailID"),
        F.col("d.ProductID"),
        F.col("d.OrderQty"),
        F.col("d.LineTotal")
    )
)

sales_join.explain("formatted")

In [ ]:
header_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

detail_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

sort_merge_join = (
    header_df.hint("merge")
    .join(
        detail_df.hint("merge"),
        on="SalesOrderID",
        how="inner"
    )
    .select(
        header_df.SalesOrderID,
        header_df.SalesOrderNumber,
        detail_df.SalesOrderDetailID,
        detail_df.ProductID,
        detail_df.OrderQty,
        detail_df.LineTotal
    )
)

sort_merge_join.explain("formatted")

In [ ]:
import time
from pyspark.sql import functions as F

header_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

detail_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

# Broadcast Hash Join
broadcast_join = (
    header_df.hint("broadcast")
    .join(detail_df, "SalesOrderID", "inner")
)

# Sort-Merge Join
sort_merge_join = (
    header_df.hint("merge")
    .join(detail_df.hint("merge"), "SalesOrderID", "inner")
)

# Aggregate the joined data to ensure both queries perform the join.
def run_query(df):
    return df.agg(
        F.count("*").alias("JoinedRows"),
        F.sum("LineTotal").alias("TotalSales")
    ).collect()

# Warm up both query strategies.
run_query(broadcast_join)
run_query(sort_merge_join)

# Measure five executions of each strategy.
results = {}

for name, df in [
    ("Broadcast Hash Join", broadcast_join),
    ("Sort-Merge Join", sort_merge_join)
]:
    timings = []

    for _ in range(5):
        start = time.perf_counter()
        output = run_query(df)
        elapsed = time.perf_counter() - start
        timings.append(elapsed)

    results[name] = {
        "TimesSeconds": timings,
        "AverageSeconds": sum(timings) / len(timings),
        "Output": output
    }

for name, result in results.items():
    print(name)
    print("Execution times:", result["TimesSeconds"])
    print("Average:", round(result["AverageSeconds"], 4), "seconds")
    print("Output:", result["Output"])
    print()